# Experiments

In [ ]:
import os
import json
import time
import shutil
import datetime
import tempfile
import jsonlines
import subprocess
import multiprocessing
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

In [ ]:
RUSE_PATH = "../target/release/Ruse"

def spawn_ruse(cmd):
    if os.fork() != 0:
        return

    proc = subprocess.Popen(cmd, close_fds=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    proc.wait()


def run_ruse_executable(args, in_background=False, ignore_output=False):
    cmd = [RUSE_PATH, "run"] + args

    if in_background:
        cmd = ["nohup"] + cmd
        ignore_output = True
    
    print(' '.join(cmd))

    if in_background:
        p = multiprocessing.Process(target=spawn_ruse, args=[cmd], daemon=True)
        p.start()
        p.join()
        return

    if ignore_output:
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    else:
        subprocess.run(cmd)

def run_ruse(tasks, results_dir, *,
             log_file = None,
             timeout=datetime.timedelta(hours=1),
             max_iterations=5,
             max_sequence_size=2,
             max_mutations=3,
             max_memory_usage="100GiB",
             workers_count=64,
             dry_run=False,
             in_background=False):
    args = [
        "-o", results_dir,
        "-t", str(int(timeout.total_seconds())),
        "--workers-count", str(workers_count),
        "--max-iterations", str(max_iterations),
        "--max-mutations", str(max_mutations),
        "--max-sequence-size", str(max_sequence_size),
        "--max-task-mem", max_memory_usage,
        '--pretty']

    if log_file is not None:
        args.append("--log")
        args.append(log_file)

    if dry_run:
        args.append("--dry-run")    
    
    for task in tasks:
        args.append("-b")
        args.append(task)
    
    run_ruse_executable(args, in_background, ignore_output=in_background or dry_run)

def parse_log(log_file):
    with jsonlines.open(log_file, "r") as reader:
        log = [line for line in reader]
    return log

def parse_results(results_dir):
    results = {
        'metadata': None,
        'tasks': []
    }
    for file in os.listdir(results_dir):
        if file == 'metadata.json':
            with open(os.path.join(results_dir, file), "r") as f:
                results['metadata'] = json.load(f)
        elif file.startswith('task_') and file.endswith('.json'):
            with open(os.path.join(results_dir, file), "r") as f:
                results['tasks'].append(json.load(f))
    return results

def get_process_pid(results_dir):
    metadata_path = os.path.join(results_dir, "metadata.json")
    timeout = time.time() + 10
    while not os.path.exists(metadata_path):
        if time.time() > timeout:
            raise Exception("Timeout waiting for metadata file")
        time.sleep(0.1)

    with open(metadata_path, "r") as f:
        metadata = json.load(f)
    return metadata['pid']

def wait_for_process(pid):
    while True:
        try:
            os.kill(pid, 0)
        except OSError:
            break
        time.sleep(0.1)

def run_experiment(name, tasks, in_background=False, **kwargs):
    os.makedirs("results", exist_ok=True)

    results_dir = os.path.join("results", f"{name}_results")
    log_file = os.path.join("results", f"{name}_log.jsonl")

    if os.path.exists(results_dir):
        shutil.rmtree(results_dir)

    run_ruse(tasks, results_dir, log_file=log_file, in_background=in_background, **kwargs)
    if not in_background:
        return parse_results(results_dir)
    else:
        return get_process_pid(results_dir)

def get_all_tasks(tasks):
    with tempfile.TemporaryDirectory() as temp_dir:
        results_dir = os.path.join(temp_dir, "results")

        run_ruse(tasks, results_dir, dry_run=True, in_background=False)
        results = parse_results(results_dir)

    tasks_parsed = {}

    oop_category_order = {
        "Full OOP": 0,
        "Primitive Objects": 1,
        "Primitive": 2,
    }

    for task in results["tasks"]:
        oop_category = task["category"].split(":")[0]
        side_effects_category = task["category"].split(":")[1]
        tasks_parsed[os.path.basename(task["path"])] = [
            oop_category_order[oop_category],
            oop_category,
            side_effects_category,
            task["path"],
            task["source"]
            # task["path"],,
        ]
        if task["error"] != None:
            raise Exception(task["error"])

    df = pd.DataFrame.from_dict(tasks_parsed, orient="index", columns=["oop_category_order", "side effects", "path", "source"])
    df.sort_values(by=["oop_category_order", "side effects"], inplace=True)
    return df

def display_df(df):
    s = df.style.set_properties(**{'text-align': 'left'})
    s.set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
    html_table = s.to_html(justify='left', notebook=True, show_dimensions=True)

    scrollable_html = f"""
<div style="max-height: 300px; overflow: auto;">
    {html_table}
</div>
"""
    display(HTML(scrollable_html))


### RQ1 - Can Ruse solve synthesis tasks from all categories correctly?

In [ ]:
tasks_paths = [
    "../tasks/benchmarks/"
]

tasks = get_all_tasks(tasks_paths)

display_df(tasks[["oop category", "side effects"]])


In [ ]:
ruse_pid = run_experiment("ruse_all_tasks", tasks_paths, in_background=True)
print(f"Ruse process pid: {ruse_pid}")

In [ ]:
results = parse_results("results/ruse_all_tasks_results")

df = pd.DataFrame(results["tasks"])
df["name"] = df["path"].apply(lambda x: os.path.basename(x).split(".")[0])

In [ ]:
print(f"{len(df[df['found'].isnull()])}/{len(df)} tasks failed")
display_df(df[df["found"].isnull()][["name", "error"]])

In [ ]:
print(f"{len(df[df['found'].notnull()])}/{len(df)} tasks passed")
display_df(df[df["found"].notnull()][["name", "found"]])
